# Detecção de Gráficos em PDFs — Comparação de 3 Métodos com Avaliação Manual

**Fluxo:**
1. Roda 3 métodos de extração sobre os PDFs em `/content/drive/MyDrive/Chandra2/PDF`.
2. Mostra, para cada PDF, **páginas originais + o que cada método extraiu**.
3. Você preenche um pequeno form: quantos gráficos / tabelas / outras imagens *realmente* existem em cada PDF.
4. Calcula o erro de cada método contra suas anotações e gera um **ranking final**.

| Método | O que faz |
|---|---|
| **1. Chandra OCR 2** | VLM, classifica blocos como `figure`/`image`/`diagram`/`table` |
| **2. PyMuPDF `get_images`** | Extrai imagens raster embutidas (PNG/JPG colados no PDF) |
| **3. PyMuPDF `get_drawings` + DBSCAN** | Agrupa caminhos vetoriais em clusters (gráficos matplotlib/ggplot) |

> Ative GPU em *Runtime → Change runtime type → GPU* (Chandra precisa).


## 1. Instalação

In [ ]:
!pip install -q "chandra-ocr[hf]" pymupdf scikit-learn pandas matplotlib pillow beautifulsoup4 ipywidgets


In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}", end='')
if torch.cuda.is_available():
    print(f"  |  {torch.cuda.get_device_name(0)}  |  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")
else:
    print("  ⚠️  ative GPU em Runtime → Change runtime type")


## 2. Configuração de pastas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

INPUT_DIR     = Path('/content/drive/MyDrive/Chandra2/PDF')
OUTPUT_ROOT   = Path('/content/drive/MyDrive/Chandra2/output')
CHANDRA_DIR   = OUTPUT_ROOT / 'chandra'
PYMUPDF_RAST  = OUTPUT_ROOT / 'pymupdf_raster'
PYMUPDF_VECT  = OUTPUT_ROOT / 'pymupdf_vector'
GT_CSV        = OUTPUT_ROOT / 'avaliacao_manual.csv'   # ground truth (você preenche)

for d in (CHANDRA_DIR, PYMUPDF_RAST, PYMUPDF_VECT):
    d.mkdir(parents=True, exist_ok=True)

# Use MAX_PDFS=None para processar todos. Ex: MAX_PDFS=5 para testar com 5 primeiros.
MAX_PDFS = None

pdfs_all = sorted(INPUT_DIR.glob('*.pdf'))
pdfs = pdfs_all if MAX_PDFS is None else pdfs_all[:MAX_PDFS]

print(f"PDFs na pasta: {len(pdfs_all)}  |  Processando: {len(pdfs)}")
for p in pdfs:
    print(f"  - {p.name}  ({p.stat().st_size/1e6:.1f} MB)")


## 3. Método 1 — Chandra OCR 2

Roda o CLI do Chandra. Primeira execução baixa ~8 GB de pesos.

In [ ]:
import subprocess, time

t0 = time.time()
proc = subprocess.Popen(
    ['chandra', str(INPUT_DIR), str(CHANDRA_DIR), '--method', 'hf'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f"\n=== Chandra concluído em {(time.time()-t0)/60:.1f} min ===")


## 4. Método 2 — PyMuPDF `get_images` (raster embutido)

In [ ]:
import fitz, io
from PIL import Image

MIN_RASTER_SIDE = 150   # ignora muito pequenas (logos, ícones)

def extract_raster(pdf_path, out_dir, min_side=MIN_RASTER_SIDE):
    out_dir.mkdir(parents=True, exist_ok=True)
    # limpa rodadas anteriores deste PDF
    for f in out_dir.glob('*'): f.unlink()

    doc = fitz.open(str(pdf_path))
    found, seen = [], set()
    for page_num, page in enumerate(doc, start=1):
        for img_info in page.get_images(full=True):
            xref = img_info[0]
            if xref in seen: continue
            seen.add(xref)
            try:
                base = doc.extract_image(xref)
                img = Image.open(io.BytesIO(base['image']))
                if min(img.size) < min_side: continue
                fname = out_dir / f"p{page_num:03d}_xref{xref}.png"
                img.save(fname)
                found.append({'page': page_num, 'xref': xref,
                              'w': img.width, 'h': img.height,
                              'path': str(fname)})
            except Exception:
                continue
    doc.close()
    return found

raster_results = {}
for pdf in pdfs:
    raster_results[pdf.name] = extract_raster(pdf, PYMUPDF_RAST / pdf.stem)
    print(f"  {pdf.name}: {len(raster_results[pdf.name])} imagens raster")


## 5. Método 3 — PyMuPDF `get_drawings` + DBSCAN (vetorial)

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

EPS_PT          = 25
MIN_SAMPLES     = 15
MIN_AREA_FRAC   = 0.03
MAX_AREA_FRAC   = 0.80
ASPECT_MIN      = 0.20
ASPECT_MAX      = 5.00
RENDER_DPI      = 150

def detect_vector(pdf_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.glob('*'): f.unlink()

    doc = fitz.open(str(pdf_path))
    found = []
    for page_num, page in enumerate(doc, start=1):
        drawings = page.get_drawings()
        if len(drawings) < MIN_SAMPLES: continue

        bboxes, centers = [], []
        for d in drawings:
            r = d.get('rect')
            if r is None: continue
            bboxes.append([r.x0, r.y0, r.x1, r.y1])
            centers.append([(r.x0+r.x1)/2, (r.y0+r.y1)/2])
        if len(centers) < MIN_SAMPLES: continue
        bboxes = np.array(bboxes); centers = np.array(centers)

        labels = DBSCAN(eps=EPS_PT, min_samples=MIN_SAMPLES).fit_predict(centers)
        page_area = page.rect.width * page.rect.height

        cluster_idx = 0
        for lab in sorted(set(labels)):
            if lab == -1: continue
            mask = labels == lab
            cb = bboxes[mask]
            x0, y0 = cb[:,0].min(), cb[:,1].min()
            x1, y1 = cb[:,2].max(), cb[:,3].max()
            w, h = x1-x0, y1-y0
            if w <= 0 or h <= 0: continue
            af = (w*h)/page_area
            if not (MIN_AREA_FRAC <= af <= MAX_AREA_FRAC): continue
            ar = w/h
            if not (ASPECT_MIN <= ar <= ASPECT_MAX): continue

            clip = fitz.Rect(x0, y0, x1, y1)
            mat = fitz.Matrix(RENDER_DPI/72, RENDER_DPI/72)
            pix = page.get_pixmap(matrix=mat, clip=clip)
            img = Image.open(io.BytesIO(pix.tobytes('png')))
            cluster_idx += 1
            fname = out_dir / f"p{page_num:03d}_c{cluster_idx:02d}.png"
            img.save(fname)
            found.append({'page': page_num,
                          'bbox': (round(x0,1), round(y0,1), round(x1,1), round(y1,1)),
                          'n_drawings': int(mask.sum()),
                          'area_frac': round(af, 3),
                          'path': str(fname)})
    doc.close()
    return found

vector_results = {}
for pdf in pdfs:
    vector_results[pdf.name] = detect_vector(pdf, PYMUPDF_VECT / pdf.stem)
    print(f"  {pdf.name}: {len(vector_results[pdf.name])} candidatos vetoriais")


## 6. Contagens do Chandra

In [ ]:
from bs4 import BeautifulSoup
from collections import Counter
import json

# Chandra HTML usa tags semânticas (não classes CSS):
#   <img>   -> figure / image / diagram (qualquer figura visual)
#   <table> -> tabela
#   <math>  -> equação
# Imagens recortadas vão como .webp soltos na própria subpasta do PDF (não em images/).

VISUAL_TYPES = ('image',)
TABLE_TYPE   = 'table'

def chandra_counts(pdf_stem):
    sub = CHANDRA_DIR / pdf_stem
    html_p = sub / f"{pdf_stem}.html"
    meta_p = sub / f"{pdf_stem}_metadata.json"
    cnt = Counter()

    # ----- HTML: tags semânticas -----
    if html_p.exists():
        soup = BeautifulSoup(html_p.read_text(encoding='utf-8'), 'html.parser')
        cnt['image']     = len(soup.find_all('img'))
        cnt['table']     = len(soup.find_all('table'))
        cnt['equation']  = len(soup.find_all('math'))
        cnt['heading']   = len(soup.find_all(['h1','h2','h3','h4','h5','h6']))
        cnt['paragraph'] = len(soup.find_all('p'))
        cnt['list']      = len(soup.find_all(['ul','ol']))

    # ----- metadata: validação cruzada -----
    if meta_p.exists():
        try:
            meta = json.loads(meta_p.read_text(encoding='utf-8'))
            cnt['_meta_total_images'] = meta.get('total_images', 0)
            cnt['_meta_num_pages']    = meta.get('num_pages', 0)
            cnt['_meta_total_chunks'] = meta.get('total_chunks', 0)
        except Exception:
            pass

    # ----- arquivos de imagem soltos na subpasta -----
    cnt['_imgs_extracted'] = (
        len(list(sub.glob('*.webp'))) +
        len(list(sub.glob('*.png')))  +
        len(list(sub.glob('*.jpg')))  +
        len(list(sub.glob('*.jpeg')))
    )
    return cnt

chandra_results = {pdf.name: chandra_counts(pdf.stem) for pdf in pdfs}

print(f"{'PDF':<70} {'<img>':>6} {'<tbl>':>6} {'<math>':>7} {'meta':>6} {'files':>6}")
print("-" * 105)
for n, c in chandra_results.items():
    print(f"  {n[:68]:<68} {c['image']:>6} {c['table']:>6} "
          f"{c['equation']:>7} {c.get('_meta_total_images', 0):>6} {c['_imgs_extracted']:>6}")


## 7. Tabela bruta — quantos cada método retornou

Esses são os números que serão comparados com a sua avaliação manual.

In [ ]:
import pandas as pd

def n_chandra_visuals(pdf_name):
    """Imagens/figuras/diagramas detectados pelo Chandra (todos viram <img> no HTML)."""
    return chandra_results[pdf_name].get('image', 0)

def n_chandra_tables(pdf_name):
    return chandra_results[pdf_name].get('table', 0)

raw_rows = []
for pdf in pdfs:
    raw_rows.append({
        'pdf': pdf.name,
        'Chandra: <img> (figure+image+diagram)': n_chandra_visuals(pdf.name),
        'Chandra: <table>': n_chandra_tables(pdf.name),
        'Chandra: total imagens extraídas': chandra_results[pdf.name]['_imgs_extracted'],
        'PyMuPDF raster': len(raster_results[pdf.name]),
        'PyMuPDF vetorial': len(vector_results[pdf.name]),
    })
raw_df = pd.DataFrame(raw_rows)
print(raw_df.to_string(index=False))

raw_df.to_csv(OUTPUT_ROOT / 'metodos_brutos.csv', index=False)
print(f"\nSalvo em: {OUTPUT_ROOT/'metodos_brutos.csv'}")


## 8. Avaliação manual — você preenche

Para cada PDF abaixo você vê:
- as páginas originais (referência),
- o que cada método extraiu (galerias compactas),
- o resumo numérico,
- e três campos para você digitar **quantos elementos visuais existem de verdade**:
  - **Gráficos** (todos os tipos: barra, linha, pizza, dispersão, etc.)
  - **Tabelas**
  - **Outras imagens** (fotos, mapas, esquemas, ilustrações que não são gráficos nem tabelas)

Clique em **💾 Salvar avaliações** no final. O arquivo fica em
`/MyDrive/Chandra2/output/avaliacao_manual.csv` e é recarregado automaticamente
nas próximas execuções (não perde o que já foi anotado).

In [ ]:
# Habilita ipywidgets no Colab
from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import math

# ----- helpers de display -----
def render_pages(pdf_path, dpi=85):
    doc = fitz.open(str(pdf_path)); pages = []
    for p in doc:
        pix = p.get_pixmap(dpi=dpi)
        pages.append(Image.open(io.BytesIO(pix.tobytes('png'))))
    doc.close(); return pages

def grid(images_with_titles, suptitle, ncols=6, fig_h_per_row=2.5):
    if not images_with_titles:
        print(f"  (vazio) {suptitle}"); return
    n = len(images_with_titles)
    cols = min(ncols, n); rows = math.ceil(n/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(2.5*cols, fig_h_per_row*rows))
    axes_iter = [axes] if n == 1 else (axes.flat if hasattr(axes,'flat') else axes)
    for i, ax in enumerate(axes_iter):
        if i < n:
            img, title = images_with_titles[i]
            ax.imshow(img); ax.set_title(title, fontsize=7)
        ax.axis('off')
    plt.suptitle(suptitle, fontsize=10); plt.tight_layout(); plt.show()

# ----- carrega ground truth existente, se houver -----
if GT_CSV.exists():
    gt_existing = pd.read_csv(GT_CSV).set_index('pdf').to_dict('index')
    print(f"📂 Carregadas avaliações anteriores de {GT_CSV.name} ({len(gt_existing)} PDFs)")
else:
    gt_existing = {}
    print("📂 Nenhuma avaliação anterior — começando do zero.")


In [ ]:
# Mostra páginas + extrações + form, PDF por PDF
all_widgets = {}

for pdf in pdfs:
    print(f"\n{'='*100}\n📄  {pdf.name}\n{'='*100}")

    # ---- páginas originais ----
    pages = render_pages(pdf)
    grid([(p, f"pág {i+1}") for i, p in enumerate(pages)],
         f"{pdf.name} — páginas originais", ncols=6, fig_h_per_row=3.5)

    # ---- resumo numérico ----
    n_chand_vis = n_chandra_visuals(pdf.name)
    n_chand_tab = n_chandra_tables(pdf.name)
    n_chand_imgs = chandra_results[pdf.name]['_imgs_extracted']
    n_rast = len(raster_results[pdf.name])
    n_vec  = len(vector_results[pdf.name])
    print(f"\n  Chandra:          {n_chand_vis} figure/image/diagram  +  {n_chand_tab} table  "
          f"(total {n_chand_imgs} imgs salvas)")
    print(f"  PyMuPDF raster:   {n_rast} imagens")
    print(f"  PyMuPDF vetorial: {n_vec} candidatos")

    # ---- galerias dos métodos ----
    sub = CHANDRA_DIR / pdf.stem / 'images'
    chandra_imgs = sorted(sub.glob('*')) if sub.exists() else []
    grid([(Image.open(p), p.name) for p in chandra_imgs],
         f"  Chandra extraiu ({len(chandra_imgs)})", ncols=6)

    grid([(Image.open(r['path']), f"p{r['page']} {r['w']}x{r['h']}")
          for r in raster_results[pdf.name]],
         f"  PyMuPDF raster ({n_rast})", ncols=6)

    grid([(Image.open(v['path']), f"p{v['page']} ({v['n_drawings']} dr)")
          for v in vector_results[pdf.name]],
         f"  PyMuPDF vetorial ({n_vec})", ncols=6)

    # ---- form de avaliação manual ----
    prev = gt_existing.get(pdf.name, {})
    w_g = widgets.IntText(value=int(prev.get('gt_graficos', 0)),
                          description='Gráficos:',
                          style={'description_width': '120px'})
    w_t = widgets.IntText(value=int(prev.get('gt_tabelas', 0)),
                          description='Tabelas:',
                          style={'description_width': '120px'})
    w_i = widgets.IntText(value=int(prev.get('gt_imagens', 0)),
                          description='Outras imagens:',
                          style={'description_width': '120px'})
    w_n = widgets.Text(value=str(prev.get('notas', '') or ''),
                       description='Notas:',
                       style={'description_width': '120px'},
                       layout=widgets.Layout(width='600px'))
    print("\n  ✏️  Quantos elementos REALMENTE existem neste PDF?")
    display(widgets.VBox([w_g, w_t, w_i, w_n]))
    all_widgets[pdf.name] = (w_g, w_t, w_i, w_n)

# ---- botão de salvar ao final ----
print("\n" + "="*100)
save_btn = widgets.Button(description='💾 Salvar avaliações',
                         button_style='success',
                         layout=widgets.Layout(width='250px', height='40px'))
save_out = widgets.Output()

def on_save(_):
    rows = []
    for pdf_name, (g, t, i, n) in all_widgets.items():
        rows.append({
            'pdf': pdf_name,
            'gt_graficos': g.value,
            'gt_tabelas':  t.value,
            'gt_imagens':  i.value,
            'gt_total':    g.value + t.value + i.value,
            'notas':       n.value,
        })
    gt_df = pd.DataFrame(rows)
    gt_df.to_csv(GT_CSV, index=False)
    with save_out:
        clear_output()
        print(f"✅ Salvo em {GT_CSV}")
        print(gt_df.to_string(index=False))

save_btn.on_click(on_save)
display(save_btn, save_out)


## 9. Ranking — qual método foi melhor?

Após preencher e clicar em salvar acima, rode esta célula. Ela calcula:

- **Erro absoluto** por PDF: `|detectado − real|` para cada método.
- **MAE** (média do erro absoluto) — quanto menor, melhor.
- **Sub-detecção** (faltou pegar) vs **super-detecção** (pegou demais / falsos positivos).

Como cada método retorna granularidades diferentes:
- *Chandra* compara `figure+image+diagram` com `gt_graficos+gt_imagens`, e `table` com `gt_tabelas`.
- *PyMuPDF raster* e *vetorial* comparam o total contra `gt_total` (gráficos+tabelas+imagens),
  porque eles não distinguem categoria.

In [ ]:
# (Re)carrega o ground truth recém-salvo
if not GT_CSV.exists():
    raise RuntimeError(f"❌ Preencha e salve a avaliação manual antes — {GT_CSV} não existe.")

gt_df = pd.read_csv(GT_CSV).set_index('pdf')
print(f"Avaliações manuais carregadas: {len(gt_df)} PDFs\n")

# Garante que só comparamos PDFs que estão tanto nos resultados quanto no GT
common = [p.name for p in pdfs if p.name in gt_df.index]
if len(common) < len(pdfs):
    missing = [p.name for p in pdfs if p.name not in gt_df.index]
    print(f"⚠️  {len(missing)} PDFs sem avaliação manual — serão ignorados:")
    for m in missing: print(f"     - {m}")

# ----- Tabela detalhada por PDF -----
detail = []
for pdf_name in common:
    gt = gt_df.loc[pdf_name]
    gt_graf_img = int(gt.gt_graficos) + int(gt.gt_imagens)
    gt_tab      = int(gt.gt_tabelas)
    gt_total    = int(gt.gt_total)

    chand_vis = n_chandra_visuals(pdf_name)
    chand_tab = n_chandra_tables(pdf_name)
    chand_total = chand_vis + chand_tab
    rast = len(raster_results[pdf_name])
    vec  = len(vector_results[pdf_name])

    detail.append({
        'pdf': pdf_name,
        'GT total': gt_total,
        '── Chandra figure/image/diagram': chand_vis, 'GT gráficos+imagens': gt_graf_img,
        '── Chandra table': chand_tab,               'GT tabelas': gt_tab,
        '── Chandra total': chand_total,
        '── PyMuPDF raster': rast,
        '── PyMuPDF vetorial': vec,
    })
detail_df = pd.DataFrame(detail)
print("Detalhe por PDF:")
print(detail_df.to_string(index=False))


In [ ]:
# ----- Métricas agregadas (MAE, viés, ranking) -----

import numpy as np

def metrics(detected, gt):
    detected, gt = np.array(detected), np.array(gt)
    err = detected - gt
    return {
        'soma_detectado':  int(detected.sum()),
        'soma_GT':         int(gt.sum()),
        'MAE':             round(float(np.abs(err).mean()), 2),
        'viés_médio':      round(float(err.mean()), 2),  # >0 super-detecta, <0 sub-detecta
        'pdfs_perfeitos':  int((err == 0).sum()),
        'pdfs_com_erro':   int((err != 0).sum()),
    }

# Vetores alinhados
gt_total_vec       = [gt_df.loc[n, 'gt_total'] for n in common]
gt_graf_img_vec    = [int(gt_df.loc[n, 'gt_graficos']) + int(gt_df.loc[n, 'gt_imagens']) for n in common]
gt_tab_vec         = [int(gt_df.loc[n, 'gt_tabelas']) for n in common]

chand_total_vec    = [n_chandra_visuals(n) + n_chandra_tables(n) for n in common]
chand_visuals_vec  = [n_chandra_visuals(n) for n in common]
chand_tables_vec   = [n_chandra_tables(n) for n in common]
rast_vec           = [len(raster_results[n]) for n in common]
vec_vec            = [len(vector_results[n]) for n in common]

results = {
    'Chandra (total)':            metrics(chand_total_vec, gt_total_vec),
    'Chandra (figure+image+diag)': metrics(chand_visuals_vec, gt_graf_img_vec),
    'Chandra (table)':            metrics(chand_tables_vec, gt_tab_vec),
    'PyMuPDF raster (total)':     metrics(rast_vec, gt_total_vec),
    'PyMuPDF vetorial (total)':   metrics(vec_vec, gt_total_vec),
}

ranking_df = pd.DataFrame(results).T
ranking_df = ranking_df.sort_values('MAE')
print("\n=== RANKING (menor MAE = mais próximo do real) ===\n")
print(ranking_df.to_string())

# Destaca o vencedor entre os métodos que medem 'total'
total_methods = {
    'Chandra (total)':            np.array(chand_total_vec),
    'PyMuPDF raster (total)':     np.array(rast_vec),
    'PyMuPDF vetorial (total)':   np.array(vec_vec),
}
gt_arr = np.array(gt_total_vec)
mae_total = {k: float(np.abs(v - gt_arr).mean()) for k, v in total_methods.items()}
winner = min(mae_total, key=mae_total.get)
print(f"\n🏆 Vencedor (comparação justa, total de elementos visuais): {winner}")
print(f"   MAE = {mae_total[winner]:.2f} elementos por PDF")

# Salva o ranking
ranking_df.to_csv(OUTPUT_ROOT / 'ranking_metodos.csv')
detail_df.to_csv(OUTPUT_ROOT / 'detalhe_por_pdf.csv', index=False)
print(f"\nSalvos:\n  - {OUTPUT_ROOT/'ranking_metodos.csv'}\n  - {OUTPUT_ROOT/'detalhe_por_pdf.csv'}")


In [ ]:
# ----- Gráfico final: viés de cada método -----
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(common))
width = 0.2

ax.bar(x - 1.5*width, gt_total_vec,    width, label='GT (total real)',    color='black')
ax.bar(x - 0.5*width, chand_total_vec, width, label='Chandra total')
ax.bar(x + 0.5*width, rast_vec,        width, label='PyMuPDF raster')
ax.bar(x + 1.5*width, vec_vec,         width, label='PyMuPDF vetorial')

ax.set_xticks(x)
ax.set_xticklabels([n.replace('.pdf','')[:15] for n in common], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('# elementos visuais detectados')
ax.set_title('Detecção total por método vs. ground truth, por PDF')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup

pdf_stem = 'W4323846702'   # ou 'W4309496579' ou 'grafico de radar'
sub = Path('/content/drive/MyDrive/Chandra2/output/chandra') / pdf_stem
html = (sub / f"{pdf_stem}.html").read_text(encoding='utf-8')
soup = BeautifulSoup(html, 'html.parser')

tables = soup.find_all('table')
print(f"Total <table>: {len(tables)}\n")
for i, t in enumerate(tables, 1):
    text = t.get_text(strip=True)[:200]
    n_rows = len(t.find_all('tr'))
    n_cells = len(t.find_all(['td','th']))
    print(f"--- TABLE {i}  (rows={n_rows}, cells={n_cells}) ---")
    print(f"  texto: {text!r}")
    print()

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup

CHANDRA_DIR = Path('/content/drive/MyDrive/Chandra2/output/chandra')

for pdf_stem in ['W4309496579', 'grafico de radar', 'artigo-histograma-capacidadeproc']:
    sub = CHANDRA_DIR / pdf_stem
    html = (sub / f"{pdf_stem}.html").read_text(encoding='utf-8')
    soup = BeautifulSoup(html, 'html.parser')

    n_imgs = len(soup.find_all('img'))
    tables = soup.find_all('table')

    print(f"\n{'='*80}")
    print(f"📄 {pdf_stem}")
    print(f"   <img>={n_imgs}   <table>={len(tables)}")
    print('='*80)

    for i, t in enumerate(tables, 1):
        text = t.get_text(strip=True)[:150]
        n_rows  = len(t.find_all('tr'))
        n_cells = len(t.find_all(['td','th']))
        print(f"  TABLE {i}  rows={n_rows}, cells={n_cells}")
        print(f"    {text!r}")

## 10. O que fazer com o resultado

- **Se um método ganhou claramente:** já é o que você usa em produção. Os outros viram fallback.
- **Se Chandra ganha em "table" mas PyMuPDF vetorial ganha em "figure":** combine os dois — o Chandra para tabelas e o vetorial para gráficos.
- **Se nenhum chega perto do GT:** olhe a coluna `viés_médio` no ranking — se for negativo, o método está sub-detectando (precisa afrouxar filtros); positivo, está super-detectando (apertar filtros).
- **Para reduzir falsos positivos do método vetorial:** suba `MIN_SAMPLES` (15→25) ou `MIN_AREA_FRAC` (0.03→0.05).
- **Para juntar gráficos quebrados em 2:** suba `EPS_PT` (25→40).


In [ ]:
from pathlib import Path

CHANDRA_DIR = Path('/content/drive/MyDrive/Chandra2/output/chandra')

print(f"Conteúdo de {CHANDRA_DIR}:\n")
if not CHANDRA_DIR.exists():
    print("❌ A pasta não existe — Chandra nunca rodou ou saiu na pasta errada.")
else:
    subs = sorted([d for d in CHANDRA_DIR.iterdir() if d.is_dir()])
    print(f"  {len(subs)} subpastas encontradas\n")
    for sub in subs[:5]:   # mostra as 5 primeiras pra não inundar
        print(f"📁 {sub.name}/")
        for f in sorted(sub.iterdir()):
            if f.is_dir():
                n = len(list(f.iterdir()))
                print(f"     📁 {f.name}/   ({n} arquivos dentro)")
            else:
                print(f"     📄 {f.name}   ({f.stat().st_size/1024:.1f} KB)")
        print()
    if len(subs) > 5:
        print(f"  ... e mais {len(subs)-5} subpastas")

In [ ]:
# Pega o primeiro PDF que tenha HTML gerado
sample_html = None
for sub in sorted(CHANDRA_DIR.iterdir()):
    if not sub.is_dir(): continue
    htmls = list(sub.glob('*.html'))
    if htmls:
        sample_html = htmls[0]
        break

if sample_html is None:
    print("❌ Nenhum .html encontrado em nenhuma subpasta.")
else:
    print(f"📄 Mostrando início de: {sample_html}\n")
    text = sample_html.read_text(encoding='utf-8')
    print(f"Tamanho total: {len(text)} chars\n")
    print("--- PRIMEIROS 3000 CHARS ---")
    print(text[:3000])
    print("\n--- TRECHO DO MEIO ---")
    mid = len(text) // 2
    print(text[mid:mid+2000])

In [ ]:
import json

sample_meta = None
for sub in sorted(CHANDRA_DIR.iterdir()):
    if not sub.is_dir(): continue
    metas = list(sub.glob('*_metadata.json')) + list(sub.glob('*.json'))
    if metas:
        sample_meta = metas[0]
        break

if sample_meta is None:
    print("❌ Nenhum JSON encontrado.")
else:
    print(f"📄 {sample_meta}\n")
    data = json.loads(sample_meta.read_text(encoding='utf-8'))
    print(f"Chaves do nível raiz: {list(data.keys()) if isinstance(data, dict) else type(data)}")
    print()
    # Mostra estrutura dos primeiros 1500 chars do JSON pretty-printed
    print(json.dumps(data, indent=2, ensure_ascii=False)[:2500])
    print("\n[...truncado]")

In [ ]:
# ===== FOCO: top-10 PDFs onde TODOS os métodos mais falham =====
import numpy as np
import pandas as pd

# Erro absoluto por método em cada PDF (em relação ao GT total)
error_per_pdf = []
for n in common:
    gt    = gt_df.loc[n, 'gt_total']
    chand = n_chandra_visuals(n) + n_chandra_tables(n)
    rast  = len(raster_results[n])
    vec   = len(vector_results[n])

    err_chand = abs(chand - gt)
    err_rast  = abs(rast  - gt)
    err_vec   = abs(vec   - gt)

    error_per_pdf.append({
        'pdf': n,
        'GT':            gt,
        'Chandra total': chand,
        'PyMuPDF raster': rast,
        'PyMuPDF vetor': vec,
        'erro_chandra':  err_chand,
        'erro_raster':   err_rast,
        'erro_vetor':    err_vec,
        'erro_SOMADO':   err_chand + err_rast + err_vec,
    })

err_df = pd.DataFrame(error_per_pdf).sort_values('erro_SOMADO', ascending=False)

print("=== TOP-10 PDFs onde TODOS os métodos mais falham ===\n")
print(err_df.head(10).to_string(index=False))

# Lista dos 10 piores
top10 = err_df.head(10)['pdf'].tolist()
print(f"\n→ {len(top10)} PDFs selecionados para análise focada")

In [ ]:
# ===== Ranking restrito aos 10 piores =====

# Mantém só os PDFs do top-10 nos vetores
mask_idx = [i for i, n in enumerate(common) if n in top10]
common_top    = [common[i] for i in mask_idx]
gt_total_t    = [gt_total_vec[i]      for i in mask_idx]
gt_graf_img_t = [gt_graf_img_vec[i]   for i in mask_idx]
gt_tab_t      = [gt_tab_vec[i]        for i in mask_idx]

chand_total_t = [chand_total_vec[i]   for i in mask_idx]
chand_vis_t   = [chand_visuals_vec[i] for i in mask_idx]
chand_tab_t   = [chand_tables_vec[i]  for i in mask_idx]
rast_t        = [rast_vec[i]          for i in mask_idx]
vec_t         = [vec_vec[i]           for i in mask_idx]

results_top = {
    'Chandra (total)':              metrics(chand_total_t, gt_total_t),
    'Chandra (figure+image+diag)':  metrics(chand_vis_t,   gt_graf_img_t),
    'Chandra (table)':              metrics(chand_tab_t,   gt_tab_t),
    'PyMuPDF raster (total)':       metrics(rast_t,        gt_total_t),
    'PyMuPDF vetorial (total)':     metrics(vec_t,         gt_total_t),
}

ranking_top = pd.DataFrame(results_top).T.sort_values('MAE')
print("\n=== RANKING RESTRITO AOS 10 PIORES ===\n")
print(ranking_top.to_string())

# Salva
ranking_top.to_csv(OUTPUT_ROOT / 'ranking_top10_falhas.csv')
err_df.to_csv(OUTPUT_ROOT / 'erros_por_pdf.csv', index=False)
print(f"\nSalvos:")
print(f"  - {OUTPUT_ROOT/'erros_por_pdf.csv'}")
print(f"  - {OUTPUT_ROOT/'ranking_top10_falhas.csv'}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(common_top))
width = 0.2

ax.bar(x - 1.5*width, gt_total_t,    width, label='GT (real)',     color='black')
ax.bar(x - 0.5*width, chand_total_t, width, label='Chandra total')
ax.bar(x + 0.5*width, rast_t,        width, label='PyMuPDF raster')
ax.bar(x + 1.5*width, vec_t,         width, label='PyMuPDF vetor')

ax.set_xticks(x)
ax.set_xticklabels([n.replace('.pdf','')[:18] for n in common_top],
                   rotation=45, ha='right', fontsize=9)
ax.set_ylabel('# elementos detectados')
ax.set_title('Top-10 PDFs com maior erro somado — comparação por método')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()